<!--
Created by Brad Delatte
Bayou Bytes
azure-sql-schema-compare
Initial architecture and implementation: 2026
-->

# Cell 1 — Notebook Settings
Set input files, output files, target schema, and DDL generation options.

In [ ]:
SCHEMA_COMPARE_FILE = "../output/schema_compare_results.xlsx"

OUTPUT_SQL_FILE = "../output/recommended_target_ddl.sql"

TARGET_SCHEMA = "dbo"

ADD_SOURCE_TRACKING_COLUMNS = True

# Cell 2 — Imports
Import required Python packages used for reading schema comparison results and generating DDL.

In [ ]:
import pandas as pd

from pathlib import Path
from datetime import datetime

print("Imports successful")

# Cell 3 — Load Schema Comparison Results
Load the previously generated schema comparison workbook.

In [ ]:
all_schema = pd.read_excel(
    SCHEMA_COMPARE_FILE,
    sheet_name="all_schema"
)

display(all_schema.head())

# Cell 4 — Define SQL Type Builder
Convert schema metadata into SQL Server data type definitions.

In [ ]:
def sql_type_from_row(row):

    data_type = str(row["data_type"]).lower()

    max_length = row["max_length"]
    precision = row["precision"]
    scale = row["scale"]

    if data_type in ["nvarchar", "nchar"]:

        if max_length == -1:
            return f"{data_type}(max)"

        return f"{data_type}({int(max_length / 2)})"

    if data_type in ["varchar", "char", "varbinary", "binary"]:

        if max_length == -1:
            return f"{data_type}(max)"

        return f"{data_type}({int(max_length)})"

    if data_type in ["decimal", "numeric"]:

        return f"{data_type}({int(precision)}, {int(scale)})"

    if data_type in ["datetime2", "datetimeoffset", "time"]:

        return f"{data_type}({int(scale)})"

    return data_type

# Cell 5 — Define Recommended Type Logic
Choose the safest recommended SQL data type when source databases disagree.

In [ ]:
def recommend_column_type(group):

    types_found = group.copy()

    types_found["sql_type"] = types_found.apply(
        sql_type_from_row,
        axis=1
    )

    unique_types = sorted(
        types_found["sql_type"]
        .dropna()
        .unique()
    )

    if len(unique_types) == 1:

        return unique_types[0]

    base_types = set(
        types_found["data_type"]
        .str.lower()
    )

    if "nvarchar" in base_types or "nchar" in base_types:

        max_len = types_found["max_length"].max()

        if max_len == -1:
            return "nvarchar(max)"

        return f"nvarchar({int(max_len / 2)})"

    if "varchar" in base_types or "char" in base_types:

        max_len = types_found["max_length"].max()

        if max_len == -1:
            return "varchar(max)"

        return f"varchar({int(max_len)})"

    if "decimal" in base_types or "numeric" in base_types:

        max_precision = int(
            types_found["precision"].max()
        )

        max_scale = int(
            types_found["scale"].max()
        )

        return f"decimal({max_precision}, {max_scale})"

    if "datetime2" in base_types:

        max_scale = int(
            types_found["scale"].max()
        )

        return f"datetime2({max_scale})"

    if "datetime" in base_types and "datetime2" in base_types:

        return "datetime2(7)"

    return "nvarchar(max)"

# Cell 6 — Build Recommended Column Definitions
Generate the recommended column definitions for each table and column.

In [ ]:
recommended_columns = (
    all_schema
    .groupby([
        "schema_name",
        "table_name",
        "column_name"
    ])
    .apply(
        lambda g: pd.Series({
            "recommended_type": recommend_column_type(g),

            "source_databases":
                ", ".join(
                    sorted(
                        g["database_name"]
                        .unique()
                    )
                ),

            "source_types":
                ", ".join(
                    sorted(
                        g.apply(
                            sql_type_from_row,
                            axis=1
                        ).unique()
                    )
                ),

            "any_nullable":
                bool(
                    g["is_nullable"].max()
                ),

            "any_identity":
                bool(
                    g["is_identity"].max()
                ),

            "min_column_id":
                g["column_id"].min()
        })
    )
    .reset_index()
    .sort_values([
        "schema_name",
        "table_name",
        "min_column_id",
        "column_name"
    ])
)

display(recommended_columns)

# Cell 7 — Generate CREATE TABLE Statements
Generate recommended SQL Server CREATE TABLE scripts for the consolidated target database.

In [ ]:
def bracket(name):

    return f"[{str(name).replace(']', ']]')}]"


ddl_scripts = []

for (
    schema_name,
    table_name
), group in recommended_columns.groupby([
    "schema_name",
    "table_name"
]):

    target_table_name = table_name

    lines = []

    lines.append(
        f"CREATE TABLE "
        f"{bracket(TARGET_SCHEMA)}."
        f"{bracket(target_table_name)}"
    )

    lines.append("(")

    column_lines = []

    if ADD_SOURCE_TRACKING_COLUMNS:

        column_lines.append(
            "    [source_database] sysname NOT NULL"
        )

        column_lines.append(
            "    ,[old_pkRecordID] int NULL"
        )

    for _, row in group.iterrows():

        column_name = row["column_name"]

        recommended_type = row["recommended_type"]

        nullable_text = "NULL"

        column_lines.append(
            f"    ,{bracket(column_name)} "
            f"{recommended_type} "
            f"{nullable_text}"
        )

    lines.extend(column_lines)

    lines.append(");")

    lines.append("GO")
    lines.append("")

    ddl_scripts.append(
        "\n".join(lines)
    )

recommended_ddl = "\n".join(ddl_scripts)

print(recommended_ddl[:4000])

# Cell 8 — Save Recommended DDL
Save the generated CREATE TABLE scripts to a SQL file.

In [ ]:
output_path = Path(OUTPUT_SQL_FILE)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

header = f"""
/*
Brad Delatte
Recommended Target DDL
Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

Source: {SCHEMA_COMPARE_FILE}

Notes:
- Includes every column found in any compared database.
- Uses safer/wider data types when conflicts are found.
- Defaults columns to NULL for migration safety.
- Review before running in production.
*/

"""

output_path.write_text(
    header + recommended_ddl,
    encoding="utf-8"
)

print(f"Saved: {output_path}")

# Cell 9 — Export Column Decision Details
Export the recommended column decisions and source type analysis to Excel.

In [ ]:
decision_file = "../output/recommended_column_decisions.xlsx"

with pd.ExcelWriter(
    decision_file,
    engine="openpyxl"
) as writer:

    recommended_columns.to_excel(
        writer,
        sheet_name="recommended_columns",
        index=False
    )

    metadata_df = pd.DataFrame([
        {
            "author": "Brad Delatte",
            "tool": "azure-sql-schema-compare",
            "created": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
    ])

    metadata_df.to_excel(
        writer,
        sheet_name="_metadata",
        index=False
    )

    writer.sheets["_metadata"].sheet_state = "hidden"

print(f"Saved: {decision_file}")

# Final Cell — Open Output Folder
Open the output folder in Windows Explorer.

In [ ]:
import os
from pathlib import Path

output_folder = Path("../output").resolve()

print(f"Opening: {output_folder}")

os.startfile(output_folder)